In [ ]:
# ════════════════════════════════════════════════════════════
# LivePortrait — driven by template_talking.mp4 (single-cell, Colab GPU)
# Animates every headshot in Drive using your talking-head template video.
#   Drive layout:  MyDrive/talking_head/
#     headshots/            person_01.png ... person_08.png
#     template_talking.mp4  driving video (head/lip motion source)
#     outputs/liveportrait/ results saved here
# ════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob, subprocess, time
from pathlib import Path

# ── Drive layout ─────────────────────────────────────────────
DRIVE_ROOT    = '/content/drive/MyDrive/talking_head'
HEADSHOTS_DIR = f'{DRIVE_ROOT}/headshots'
OUTPUTS_DIR   = f'{DRIVE_ROOT}/outputs'
TEMPLATE_MP4  = f'{DRIVE_ROOT}/template_talking.mp4'
MODELS_DIR    = '/content/models'   # local to the Colab session (fast)

LP_REPO = Path(MODELS_DIR) / 'liveportrait'
LP_OUT  = Path(OUTPUTS_DIR) / 'liveportrait'
LP_OUT.mkdir(parents=True, exist_ok=True)

HF_REPO = 'KlingTeam/LivePortrait'
WEIGHT_FILES = [
    'liveportrait/base_models/appearance_feature_extractor.pth',
    'liveportrait/base_models/motion_extractor.pth',
    'liveportrait/base_models/spade_generator.pth',
    'liveportrait/base_models/warping_module.pth',
    'liveportrait/retargeting_models/stitching_retargeting_module.pth',
    'liveportrait/landmark.onnx',
]

def _fmt(s):
    m, s = divmod(int(s), 60); h, m = divmod(m, 60)
    return f'{h}h {m:02d}m {s:02d}s' if h else (f'{m}m {s:02d}s' if m else f'{s}s')
def _step(msg): print(f'\n[{time.strftime("%H:%M:%S")}] {msg}', flush=True)

# ── Headshots + template checks ──────────────────────────────
headshots = sorted(glob.glob(f'{HEADSHOTS_DIR}/*.png') + glob.glob(f'{HEADSHOTS_DIR}/*.jpg'))
if not headshots:
    raise RuntimeError(f'No headshots found in {HEADSHOTS_DIR}')
if not os.path.exists(TEMPLATE_MP4):
    raise RuntimeError(f'Driver video not found: {TEMPLATE_MP4}')
print(f'Found {len(headshots)} headshots.')
print(f'Driver template: {TEMPLATE_MP4}')

# ── Setup (clone + install) ──────────────────────────────────
if not (LP_REPO / '.setup_done').exists():
    _step('SETUP — cloning LivePortrait + installing deps')
    if not LP_REPO.exists():
        subprocess.run(['git', 'clone', 'https://github.com/KwaiVGI/LivePortrait', str(LP_REPO)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '-r', str(LP_REPO / 'requirements_base.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'onnxruntime', 'transformers==4.38.0', 'huggingface_hub'], check=True)
    (LP_REPO / '.setup_done').touch()
    print('  Setup done.')
else:
    print(f'[{time.strftime("%H:%M:%S")}] Setup already done, skipping.')

# ── Weights (~600 MB, once) ──────────────────────────────────
if not (LP_REPO / '.weights_done').exists():
    _step('WEIGHTS — downloading 6 files from HuggingFace')
    from huggingface_hub import hf_hub_download
    weights_dir = LP_REPO / 'pretrained_weights'
    weights_dir.mkdir(parents=True, exist_ok=True)
    for f in WEIGHT_FILES:
        dest = weights_dir / f
        dest.parent.mkdir(parents=True, exist_ok=True)
        if not dest.exists():
            print(f'  {Path(f).name} ...', end=' ', flush=True)
            t0 = time.time()
            hf_hub_download(repo_id=HF_REPO, filename=f, local_dir=str(weights_dir))
            print(f'{dest.stat().st_size/1024/1024:.0f} MB in {_fmt(time.time()-t0)}')
        else:
            print(f'  already have {Path(f).name}')
    (LP_REPO / '.weights_done').touch()
else:
    print(f'[{time.strftime("%H:%M:%S")}] Weights already downloaded, skipping.')

# ── InsightFace buffalo_l (det + landmark) ───────────────────
# LivePortrait crops faces with InsightFace, which only needs det_10g.onnx
# + 2d106det.onnx. But FaceAnalysis loads EVERY .onnx in the folder on init,
# so a corrupt/auto-downloaded 1k3d68.onnx breaks startup. Keep the folder
# to exactly these two valid files from the HF weights repo. Idempotent.
from huggingface_hub import hf_hub_download
_buffalo = LP_REPO / 'pretrained_weights' / 'insightface' / 'models' / 'buffalo_l'
_needed  = ['det_10g.onnx', '2d106det.onnx']
def _ok(p): return p.exists() and p.stat().st_size > 1_000_000   # stubs are <1 KB
if _buffalo.exists():
    for p in _buffalo.glob('*.onnx'):       # drop corrupt or unused onnx
        if p.name not in _needed or not _ok(p):
            p.unlink()
_buffalo.mkdir(parents=True, exist_ok=True)
if not all(_ok(_buffalo / n) for n in _needed):
    _step('INSIGHTFACE — downloading buffalo_l det + landmark models')
    for n in _needed:
        if not _ok(_buffalo / n):
            print(f'  {n} ...', end=' ', flush=True)
            hf_hub_download(repo_id=HF_REPO,
                            filename=f'insightface/models/buffalo_l/{n}',
                            local_dir=str(LP_REPO / 'pretrained_weights'))
            print(f'{(_buffalo/n).stat().st_size/1024/1024:.0f} MB')
print(f'[{time.strftime("%H:%M:%S")}] buffalo_l ready: '
      f'{sorted(p.name for p in _buffalo.glob("*.onnx"))}')

# ── Inference (template_talking.mp4 as driver) ───────────────
# --flag_crop_driving_video crops the template to its face region so the
# motion transfers cleanly to each still headshot.
_step(f'INFERENCE — {len(headshots)} headshots  |  driver: template_talking.mp4')
total_start = time.time()

for headshot in headshots:
    name     = Path(headshot).stem
    out_path = str(LP_OUT / f'{name}_talking.mp4')
    if os.path.exists(out_path):
        print(f'  skip {name} (already exists)')
        continue
    work_dir = str(LP_OUT / name)
    os.makedirs(work_dir, exist_ok=True)
    cmd = [sys.executable, str(LP_REPO / 'inference.py'),
           '--source', headshot,
           '--driving', TEMPLATE_MP4,
           '--output_dir', work_dir,
           '--driving_multiplier', '1.2',
           '--flag_crop_driving_video']
    print(f'  [{time.strftime("%H:%M:%S")}] {name} ...', end=' ', flush=True)
    t0     = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(LP_REPO),
                            env={**os.environ, 'PYTHONUTF8': '1'})
    elapsed = time.time() - t0
    if result.returncode != 0:
        print(f'ERROR\n{result.stderr[-500:]}')
        continue
    mp4s = sorted(glob.glob(os.path.join(work_dir, '**', '*.mp4'), recursive=True),
                  key=os.path.getmtime, reverse=True)
    if mp4s:
        os.replace(mp4s[0], out_path)
        print(f'done in {_fmt(elapsed)}  ({os.path.getsize(out_path)/1024/1024:.2f} MB)')
    else:
        print('WARNING: no mp4 found')

print(f'\n{"="*50}')
print(f'LivePortrait done. Total: {_fmt(time.time()-total_start)}')
print(f'Outputs: {LP_OUT}')


In [ ]:

# ── Drive layout ─────────────────────────────────────────────
DRIVE_ROOT    = '/content/drive/MyDrive/talking_head'
HEADSHOTS_DIR = f'{DRIVE_ROOT}/headshots'
OUTPUTS_DIR   = f'{DRIVE_ROOT}/outputs'
TEMPLATE_MP4  = f'{DRIVE_ROOT}/template_talking.mp4'
MODELS_DIR    = '/content/models'   # local to the Colab session (fast)

LP_REPO = Path(MODELS_DIR) / 'liveportrait'
LP_OUT  = Path(OUTPUTS_DIR) / 'liveportrait'
LP_OUT.mkdir(parents=True, exist_ok=True)

HF_REPO = 'KlingTeam/LivePortrait'
WEIGHT_FILES = [
    'liveportrait/base_models/appearance_feature_extractor.pth',
    'liveportrait/base_models/motion_extractor.pth',
    'liveportrait/base_models/spade_generator.pth',
    'liveportrait/base_models/warping_module.pth',
    'liveportrait/retargeting_models/stitching_retargeting_module.pth',
    'liveportrait/landmark.onnx',
]

def _fmt(s):
    m, s = divmod(int(s), 60); h, m = divmod(m, 60)
    return f'{h}h {m:02d}m {s:02d}s' if h else (f'{m}m {s:02d}s' if m else f'{s}s')
def _step(msg): print(f'\n[{time.strftime("%H:%M:%S")}] {msg}', flush=True)

# ── Headshots + template checks ──────────────────────────────
headshots = sorted(glob.glob(f'{HEADSHOTS_DIR}/*.png') + glob.glob(f'{HEADSHOTS_DIR}/*.jpg'))
if not headshots:
    raise RuntimeError(f'No headshots found in {HEADSHOTS_DIR}')
if not os.path.exists(TEMPLATE_MP4):
    raise RuntimeError(f'Driver video not found: {TEMPLATE_MP4}')
print(f'Found {len(headshots)} headshots.')
print(f'Driver template: {TEMPLATE_MP4}')

# ── Setup (clone + install) ──────────────────────────────────
if not (LP_REPO / '.setup_done').exists():
    _step('SETUP — cloning LivePortrait + installing deps')
    if not LP_REPO.exists():
        subprocess.run(['git', 'clone', 'https://github.com/KwaiVGI/LivePortrait', str(LP_REPO)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '-r', str(LP_REPO / 'requirements_base.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'onnxruntime', 'transformers==4.38.0', 'huggingface_hub'], check=True)
    (LP_REPO / '.setup_done').touch()
    print('  Setup done.')
else:
    print(f'[{time.strftime("%H:%M:%S")}] Setup already done, skipping.')

# ── Weights (~600 MB, once) ──────────────────────────────────
if not (LP_REPO / '.weights_done').exists():
    _step('WEIGHTS — downloading 6 files from HuggingFace')
    from huggingface_hub import hf_hub_download
    weights_dir = LP_REPO / 'pretrained_weights'
    weights_dir.mkdir(parents=True, exist_ok=True)
    for f in WEIGHT_FILES:
        dest = weights_dir / f
        dest.parent.mkdir(parents=True, exist_ok=True)
        if not dest.exists():
            print(f'  {Path(f).name} ...', end=' ', flush=True)
            t0 = time.time()
            hf_hub_download(repo_id=HF_REPO, filename=f, local_dir=str(weights_dir))
            print(f'{dest.stat().st_size/1024/1024:.0f} MB in {_fmt(time.time()-t0)}')
        else:
            print(f'  already have {Path(f).name}')
    (LP_REPO / '.weights_done').touch()
else:
    print(f'[{time.strftime("%H:%M:%S")}] Weights already downloaded, skipping.')

# ── InsightFace buffalo_l (det + landmark) ───────────────────
# LivePortrait crops faces with InsightFace, which only needs det_10g.onnx
# + 2d106det.onnx. But FaceAnalysis loads EVERY .onnx in the folder on init,
# so a corrupt/auto-downloaded 1k3d68.onnx breaks startup. Keep the folder
# to exactly these two valid files from the HF weights repo. Idempotent.
from huggingface_hub import hf_hub_download
_buffalo = LP_REPO / 'pretrained_weights' / 'insightface' / 'models' / 'buffalo_l'
_needed  = ['det_10g.onnx', '2d106det.onnx']
def _ok(p): return p.exists() and p.stat().st_size > 1_000_000   # stubs are <1 KB
if _buffalo.exists():
    for p in _buffalo.glob('*.onnx'):       # drop corrupt or unused onnx
        if p.name not in _needed or not _ok(p):
            p.unlink()
_buffalo.mkdir(parents=True, exist_ok=True)
if not all(_ok(_buffalo / n) for n in _needed):
    _step('INSIGHTFACE — downloading buffalo_l det + landmark models')
    for n in _needed:
        if not _ok(_buffalo / n):
            print(f'  {n} ...', end=' ', flush=True)
            hf_hub_download(repo_id=HF_REPO,
                            filename=f'insightface/models/buffalo_l/{n}',
                            local_dir=str(LP_REPO / 'pretrained_weights'))
            print(f'{(_buffalo/n).stat().st_size/1024/1024:.0f} MB')
print(f'[{time.strftime("%H:%M:%S")}] buffalo_l ready: '
      f'{sorted(p.name for p in _buffalo.glob("*.onnx"))}')

# ── Inference (template_talking.mp4 as driver) ───────────────
# --flag_crop_driving_video crops the template to its face region so the
# motion transfers cleanly to each still headshot.
_step(f'INFERENCE — {len(headshots)} headshots  |  driver: template_talking.mp4')
total_start = time.time()

for headshot in headshots:
    name     = Path(headshot).stem
    out_path = str(LP_OUT / f'{name}_talking.mp4')
    if os.path.exists(out_path):
        print(f'  skip {name} (already exists)')
        continue
    work_dir = str(LP_OUT / name)
    os.makedirs(work_dir, exist_ok=True)
    cmd = [sys.executable, str(LP_REPO / 'inference.py'),
           '--source', headshot,
           '--driving', TEMPLATE_MP4,
           '--output_dir', work_dir,
           '--driving_multiplier', '1.2',
           '--flag_crop_driving_video']
    print(f'  [{time.strftime("%H:%M:%S")}] {name} ...', end=' ', flush=True)
    t0     = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(LP_REPO),
                            env={**os.environ, 'PYTHONUTF8': '1'})
    elapsed = time.time() - t0
    if result.returncode != 0:
        print(f'ERROR\n{result.stderr[-500:]}')
        continue
    mp4s = sorted(glob.glob(os.path.join(work_dir, '**', '*.mp4'), recursive=True),
                  key=os.path.getmtime, reverse=True)
    if mp4s:
        os.replace(mp4s[0], out_path)
        print(f'done in {_fmt(elapsed)}  ({os.path.getsize(out_path)/1024/1024:.2f} MB)')
    else:
        print('WARNING: no mp4 found')

print(f'\n{"="*50}')
print(f'LivePortrait done. Total: {_fmt(time.time()-total_start)}')
print(f'Outputs: {LP_OUT}')
